<a href="https://colab.research.google.com/github/koewilliams5/DI-Bootcamp/blob/main/agentic_ai_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agentic AI and RAGs ?

Using only open source frameworks.
- Tiny KB with FAISS + FakeEmbeddings
- Free Wikipedia utility (no API keys)
- Rule-based planner
- Stub summarizer with optional tiny HF model (sshleifer/tiny-gpt2)
Run all cells top to bottom in Colab.

In [ ]:
!pip install -q langchain langchain-community faiss-cpu wikipedia transformers accelerate sentencepiece

## 1) Build the KB retriever

In [ ]:
!pip install -q faiss-cpu
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import FakeEmbeddings

kb_docs = [
    Document(
        page_content="""
Agentic systems reason step-by-step about which tools to call instead of invoking tools blindly.
Their core loop is: (1) interpret the user goal, (2) inspect available context, (3) decide whether tools are needed,
(4) call one or more tools in a planned sequence, and (5) synthesize an answer grounded in the tool results.

Key principles for agentic behavior:
- Prefer direct answering from retrieved or provided context when sufficient; avoid unnecessary tool calls.
- Choose the simplest tool or minimal set of tools that can satisfy the user’s goal.
- When multiple tools are needed, sketch a short plan (ordered tool calls with dependencies) before executing.
- After each tool call, update an internal scratchpad and re-evaluate whether more tools are still necessary.
- Handle tool failures gracefully: inspect error messages, adjust inputs, or fall back to alternative tools.

Agentic systems must also enforce safety and constraints:
- Do not call tools that violate policy or appear unsafe given the user’s intent.
- Throttle repetitive or redundant calls that would waste resources without improving the answer.
- Be explicit in reasoning about trade-offs (e.g., precision vs. latency) when picking tools.
""".strip(),
        metadata={"source": "kb:agentic_concept"},
    ),
    Document(
        page_content="""
Retrievers fetch grounding passages from a knowledge base and are the primary interface to internal documents.
Given a user query, the system should: (1) normalize and possibly expand the query, (2) retrieve top-k candidates,
(3) read them carefully, and (4) base the answer primarily on those passages.

Best practices for using retrievers:
- Start with a moderately small k (e.g., 3–10) to keep context focused; increase k only if evidence is clearly missing.
- Reformulate the query if results look noisy or off-topic (e.g., add constraints, synonyms, or key entities).
- Use metadata filters (dates, document types, domains) when available to narrow down to the most relevant sources.
- When multiple passages agree, highlight the shared core facts; when they conflict, surface the disagreement explicitly.

Citing retriever outputs:
- Quote or paraphrase the most important sentences and explicitly cite their source identifiers (e.g., document IDs).
- Avoid cherry-picking single passages that contradict the majority of evidence without acknowledging the discrepancy.
- If retrieved passages do not directly support a precise answer, say so instead of guessing, and suggest follow-up queries.
""".strip(),
        metadata={"source": "kb:retrievers"},
    ),
    Document(
        page_content="""
Wikipedia is a broad-coverage, free fallback when the curated knowledge base lacks coverage or appears incomplete.
It should not be the first resort when high-quality internal documents exist, but can complement them for general facts,
background context, or definitions.

Guidelines for using Wikipedia:
- First attempt retrieval over the internal KB; only query Wikipedia if internal retrieval yields little or no relevant evidence.
- Use Wikipedia mainly for widely accepted, non-controversial information (basic facts, dates, definitions, high-level overviews).
- For critical or sensitive topics (e.g., medical, legal, or contentious political claims), treat Wikipedia as a starting point,
  not as the final authority; cross-check with more authoritative sources when possible.
- Pay attention to section structure: lead sections usually contain stable, high-level facts; deeper sections may be more speculative.

Communicating Wikipedia usage to users:
- Clearly indicate when information is derived from Wikipedia or similar external encyclopedic sources.
- Prefer concise, well-supported statements over niche details that rely on poorly sourced or disputed claims.
- If a Wikipedia article appears inconsistent or low quality, either omit those details or explicitly flag the uncertainty.
""".strip(),
        metadata={"source": "kb:wikipedia_tip"},
    ),
    Document(
        page_content="""
When evidence is thin, ambiguous, or conflicting, the system must be transparent about uncertainty instead of fabricating detail.
Honesty requires separating what is directly supported by sources from what is inferred, and clearly expressing confidence levels.

Practical rules for honest behavior:
- Always distinguish between: (a) facts directly supported by retrieved passages, (b) reasonable inferences, and (c) speculation.
- If key information is missing, explicitly say that the available sources do not answer the question fully.
- Do not invent citations, URLs, names, or numbers to fill gaps; if you cannot support them, do not present them as factual.
- When sources conflict, summarize each position, highlight the disagreement, and avoid taking a definitive stance without justification.

Helpful ways to express uncertainty:
- Use phrases like “the retrieved sources indicate…”, “based on the available evidence…”, or “the documents do not specify…”.
- Suggest concrete next steps: clarifying the question, expanding or narrowing retrieval, or consulting domain experts or
  authoritative references.
- Prefer being conservatively accurate over confidently wrong, even if that means providing a partial answer.
""".strip(),
        metadata={"source": "kb:honesty"},
    ),
    Document(
        page_content="""
Answers should be concise, focused, and well-structured, typically within 2–4 sentences for straightforward questions.
Lead with the main conclusion, then briefly justify it using the most relevant evidence and clearly marked citations.

Style and structure guidelines:
- Directly answer the user’s question in the first sentence whenever possible.
- Avoid long preambles, restating the entire question, or narrating internal reasoning unless explicitly asked.
- Use clear, concrete language; minimize jargon and define necessary technical terms in simple words.
- For complex topics, provide a short high-level summary first, then optionally a brief, structured breakdown (e.g., bullet points).

Citation and formatting:
- Cite the minimal set of sources that substantively support the key claims, rather than listing all retrieved documents.
- Integrate citations inline, near the claims they support, to make it easy to trace evidence.
- Keep formatting clean: short paragraphs, occasional bullets for lists, and no unnecessary verbosity.

When to expand beyond 2–4 sentences:
- If the user explicitly asks for a detailed explanation, tutorial, or step-by-step reasoning.
- If the topic is safety-critical and requires more context, caveats, or assumptions to avoid misinterpretation.
In all cases, maintain clarity and relevance as the primary goals.
""".strip(),
        metadata={"source": "kb:style"},
    ),
]

embeddings = FakeEmbeddings(size=256)
vs = FAISS.from_documents(kb_docs, embeddings)
retriever = vs.as_retriever(search_kwargs={"k": 3})
print("KB ready with", len(kb_docs), "docs")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 84.2 MB/s eta 0:00:00
KB ready with 5 docs


## 2) Open source external tool: Wikipedia search

In [ ]:
!pip install -q wikipedia
import wikipedia
from wikipedia.exceptions import WikipediaException, PageError, DisambiguationError
import time
import json # Added to catch JSONDecodeError

# Set Wikipedia language globally for the wikipedia library
wikipedia.set_lang('en')

def wiki_search(query: str, k: int = 2, retries: int = 5, delay: int = 2, doc_content_chars_max: int = 1000):
    snippets = []
    last_error = None # Track the last error
    for i in range(retries):
        print(f"Attempt {i+1} for query: '{query}'")
        try:
            # Search for page titles
            search_titles = wikipedia.search(query, results=k)
            if not search_titles:
                print(f"Attempt {i+1}: No search results found for '{query}'.")
                if i < retries - 1:
                    time.sleep(delay)
                continue

            for title in search_titles:
                try:
                    # Get the page object
                    page = wikipedia.page(title, auto_suggest=False, redirect=True)
                    # Get a summary (LangChain's wrapper often uses summary, not full content)
                    summary_text = wikipedia.summary(title, sentences=2, auto_suggest=False, redirect=True) # sentences=2 often used for snippets

                    # Truncate content if it exceeds max chars
                    if doc_content_chars_max and len(summary_text) > doc_content_chars_max:
                        summary_text = summary_text[:doc_content_chars_max] + "..."

                    snippets.append({
                        'title': page.title,
                        'summary': summary_text
                    })
                except PageError:
                    print(f"Attempt {i+1}: PageError for title '{title}'. Skipping this title.")
                    continue
                except DisambiguationError as de:
                    # If it's a disambiguation page, try the first option
                    print(f"Attempt {i+1}: DisambiguationError for '{title}'. Trying first option: {de.options[0]}")
                    try:
                        page = wikipedia.page(de.options[0], auto_suggest=False, redirect=True)
                        summary_text = wikipedia.summary(de.options[0], sentences=2, auto_suggest=False, redirect=True)
                        if doc_content_chars_max and len(summary_text) > doc_content_chars_max:
                            summary_text = summary_text[:doc_content_chars_max] + "..."
                        snippets.append({
                            'title': page.title,
                            'summary': summary_text
                        })
                    except json.JSONDecodeError as sub_e:
                        print(f"Attempt {i+1}: JSONDecodeError for disambiguated title '{de.options[0]}': {sub_e}")
                        last_error = sub_e
                        continue
                    except Exception as sub_e:
                        print(f"Attempt {i+1}: Failed to get content for disambiguated title '{de.options[0]}': {sub_e}")
                        last_error = sub_e # Capture the error
                        continue
                except json.JSONDecodeError as page_e: # Catch JSONDecodeError specifically here
                    print(f"Attempt {i+1}: JSONDecodeError when processing page '{title}': {page_e}. (from wikipedia.page or .summary)")
                    last_error = page_e
                    continue
                except Exception as page_e:
                    print(f"Attempt {i+1}: General error processing page '{title}': {page_e}")
                    last_error = page_e # Capture the error
                    continue

            if snippets:
                return snippets, None # Success
            else:
                print(f"Attempt {i+1}: No snippets collected for '{query}' after processing titles.")

        except WikipediaException as e:
            print(f"Attempt {i+1} failed with WikipediaException: {e}")
            last_error = e # Capture the error
        except json.JSONDecodeError as e: # Catch JSONDecodeError specifically for wikipedia.search
            print(f"Attempt {i+1} failed with JSONDecodeError during search for '{query}': {e}. Wikipedia API might be returning malformed data or an error page.")
            last_error = e # Capture the error
        except Exception as e:
            print(f"Attempt {i+1} failed with general Exception (external library/network issue): {e}")
            last_error = e # Capture the error

        # If we reach here, an error occurred or no snippets were found, so wait and retry
        if i < retries - 1:
            time.sleep(delay)
        else:
            # If all retries fail, return the last encountered error
            return [], f"Failed to retrieve Wikipedia snippets after {retries} retries. Last error: {str(last_error) if last_error else 'Unknown error'}."

    return [], f"Failed to retrieve Wikipedia snippets after multiple retries. Last error: {str(last_error) if last_error else 'Unknown error'}."

result, error = wiki_search('Python programming')
if error:
    print(f"Test search failed: {error}")
else:
    if result:
        print(f"First snippet title: {result[0]['title']}, summary: {result[0]['summary'][:50]}...")
    else:
        print("Test search returned no snippets but no error was reported.")

Attempt 1 for query: 'Python programming'
First snippet title: Python (programming language), summary: Python is a high-level, general-purpose programmin...


## 3) Simple planner (rule-based)

In [ ]:
kb_keywords = ['agentic', 'retriever', 'citation', 'ground', 'transparen', 'honest']

def plan(question: str):
    if any(keyword in question.lower() for keyword in kb_keywords):
        return {'action': 'kb'}
    else:
        return {'action': 'wiki'}

print(plan('How to ground answers?'))
print(plan('Who created Python?'))

{'action': 'kb'}
{'action': 'wiki'}


## 4) Answer function with stub or tiny HF model

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.chat_models.fake import FakeListChatModel
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

# Définit le modèle de prompt pour l'assistant agentique
prompt = ChatPromptTemplate.from_template("You are a helpful agentic assistant. Use the given context and wiki snippets."
    "If there is little evidence, say so and suggest a follow-up query."
    "Cite sources like [kb:doc1] or [wiki:Title]."
    "Question: {question}"
    "Context:{context}"
    "Wiki:{wiki}"
    "Answer:"
)

# Configure et retourne un générateur de texte basé sur TinyLlama
def get_tiny_generator(model_id: str = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'):
    # Charge le tokenizer et le modèle pré-entraîné
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id)
    # Crée un pipeline de génération de texte
    return pipeline('text-generation', model=model, tokenizer=tok, torch_dtype=torch.float32, device_map='auto')

# Utilise le modèle TinyLlama pour résumer ou générer du texte
def summarize_with_tiny(prompt_text: str, max_new_tokens: int = 120):
    gen = get_tiny_generator()
    out = gen(
        prompt_text,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_k=50,
        top_p=0.95
    )
    # Extrait la partie générée du texte
    full = out[0]["generated_text"]
    completion = full[len(prompt_text):].strip()
    return completion

# Fonction principale pour répondre à une question
def answer_question(question: str, use_tiny_model: bool = True):
    # Étape 1: Planification - décide d'utiliser la KB ou Wikipedia
    pl = plan(question)

    # Étape 2: Récupération des données
    docs = []
    wiki_snips = []
    wiki_err = None

    if pl['action']=='kb':
        # Récupère les documents de la base de connaissances interne
        docs = retriever.invoke(question)
    elif pl['action']=='wiki':
        # Effectue une recherche Wikipedia si le plan le requiert
        wiki_snips, wiki_err = wiki_search(question)

    # Formate le contexte pour le prompt
    context_text = '\n'.join([f"[{d.metadata.get('source')}] {d.page_content}" for d in docs]) or 'No KB context.'
    wiki_text = '\n'.join([f"[wiki:{s['title']}] {s['summary']}" for s in wiki_snips]) or 'No wiki snippets.'

    # Étape 3: Génération de la réponse
    # Formate les messages pour le modèle de génération
    messages = prompt.format_messages(question=question, context=context_text, wiki=wiki_text)

    # Choisit entre le modèle TinyLlama ou une réponse 'stub' rapide
    if use_tiny_model:
        final_answer = summarize_with_tiny(messages[0].content) # Utilise TinyLlama pour une réponse réelle
    else:
        stub = FakeListChatModel(responses=['Stub summary based on provided context and wiki snippets.']) # Réponse 'stub' pour le débogage/rapidité
        final_answer = stub.invoke(messages).content

    # Retourne le plan, les sources utilisées et la réponse finale
    return {
        'plan': pl,
        'kb_sources': [d.metadata.get('source') for d in docs],
        'wiki_sources': [s.get('title') for s in wiki_snips],
        'wiki_error': wiki_err,
        'answer': final_answer,
    }

## 5) Quick check on sample questions

In [ ]:
# Définit une liste de questions de test
tests = [
    "What are the key principles for agentic behavior?",
    "Who invented Python?"
]

# Itère sur chaque question et affiche la réponse générée
for q in tests:
    # Appelle la fonction answer_question, en activant le modèle TinyLlama
    res = answer_question(q, use_tiny_model=True)
    print('Q:', q) # Affiche la question
    print('Plan:', res['plan']) # Affiche le plan d'action (KB ou Wiki)
    print('KB sources:', res['kb_sources']) # Affiche les sources de la base de connaissances
    print('Wiki sources:', res['wiki_sources']) # Affiche les sources Wikipedia
    print('Answer:', res['answer']) # Affiche la réponse générée

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What are the key principles for agentic behavior?
Plan: {'action': 'kb'}
KB sources: ['kb:honesty', 'kb:style', 'kb:agentic_concept']
Wiki sources: []
Answer: “It is a useful tool to simplify the process of answering questions related to the given context. By utilizing the given context and wiki snippets, the tool can provide a clear and concise explanation of the key principles for agentic behavior, as well as the key principles for honest behavior. The tool also provides practical rules for honest behavior, including distinguishing between facts directly supported by retrieved passages, reasonable inferences, and speculation, using clear phrases to express uncertainty, and avoiding unnecessary tool calls. By adopting these principles, agentic systems can reason step-by-
Attempt 1: General error processing page 'Python (programming language)': Expecting value: line 1 column 1 (char 0)
Attempt 1: General error processing page 'Monty Python's Flying Circus': Expecting value: line 1 c

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Who invented Python?
Plan: {'action': 'wiki'}
KB sources: []
Wiki sources: ['Python (programming language)', "Monty Python's Flying Circus"]
Answer: Python is an open-source, high-level programming language developed at the University of Cambridge in the United Kingdom. It is known for its ease-of-use, readability, and popularity in the scientific and data analytics communities. Python's syntax and semantics are designed to be as concise and readable as possible, and it is one of the most popular programming languages in the world.
